In [ ]:
# Practice Problems Day 5

### Author: Joanna Bieri (solutions)

## Introduction to Machine Learning

#### University of Redlands - DATA 301
#### Prof: Joanna Bieri [joanna_bieri@redlands.edu](mailto:joanna_bieri@redlands.edu)
#### Day 5 Practice - SOLUTIONS
#### [Class Website](https://joannabieri.com/machine_learning.html)

---

**Reading:** Geron, chapter 3, the section on **Performance Measures**

GOALS:

1. Get the confusion matrix, precision, and recall out of a real classifier, and say what they mean for real people.
2. Pull the probabilities out with `predict_proba` and move the threshold yourself.
3. Compute ROC AUC and average precision, and know which one to report.
4. Make every choice on a validation set, and open the test set once, at the end.

**How much of this should you write yourself?** Every problem says. The short version: Problem 1 gives you most of the code with blanks. Problems 2 and 3 give you the steps and you write the lines. Problem 5 just tells you what to do. That is on purpose, so by the end you have written the whole Day 5 recipe with your own hands. If you get stuck, the "Asking AI about this code" box in the Day 5 notes has the three prompts that will actually help.

**These are my solutions.** Try every problem yourself first, then come here to check your work. There is more than one right way to write the code, and your words will be different from mine, that is fine! If your numbers do not match, look for the difference before you look at my code.

---

## Problem 1: A cancer screening model (fill in the blanks)

sklearn ships a real dataset: 569 breast tumors, 30 measurements each, and whether the tumor turned out to be **malignant** or **benign**. This is a screening problem, so the mistakes are not symmetric. Missing a malignant tumor is much worse than sending a benign one for a follow up scan.

**What to copy, what to write:** The code below is complete except for the blanks marked `____`. Fill them in. Each blank has a comment saying what goes there. Do not paste the wine code over the top of it, the point is to see that the same recipe works on new data.

**There is a trap in this dataset, and it is in the first blank.** sklearn stores the label as 0 for malignant and 1 for benign. That is backwards from what we want: for a screening test the **positive** class, the thing we are trying to find, is malignant. If you leave the label alone, "recall" will mean "share of benign tumors we found," which is not the question. So we build our own `y` where 1 means malignant.

**The three sets.** Same as the notes: the test set is split off first and put away, then a validation set is split off the training data. Everything in Problems 2, 3, and 5a happens on the **validation** set. The test set gets opened once, in 5c.

**1a.** Fill in the blanks and run it.

**1b.** Print the confusion matrix. Then write out, in words, what each of the four cells means **for a patient**. "False positive" is not an answer. "A benign tumor that the model flagged, so the patient gets a scan they did not need" is.

**1c.** Compare the accuracy to the always-no baseline (the share of benign tumors in the validation set). How much did the model actually do?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

# Step 1: load the data
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
print("target_names:", data.target_names)      # index 0 is malignant, index 1 is benign

# Step 2: build the label so that 1 means MALIGNANT (the thing we want to find)
y = pd.Series((data.target == 0).astype(int))   # a Series, like the wine label, so .values works the same way
print("share malignant:", round(y.mean(), 3))

# Step 3: split the TEST set off first, stratified, 25 percent. Then put it away.
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y,
    test_size=0.25,
    stratify=y,
    random_state=42)

# Step 4: split a VALIDATION set off the training data, same way
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full,
    test_size=0.25,
    stratify=y_train_full,
    random_state=42)

# Step 5: scale on the training data only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

# Step 6: fit, then predict on the VALIDATION set
model = LogisticRegression(max_iter=5000)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_valid_scaled)

print("accuracy:", round(accuracy_score(y_valid, y_pred), 4))
print("always-no baseline:", round(1 - y_valid.mean(), 4))

In [ ]:
cm = confusion_matrix(y_valid, y_pred)
print(cm)
tn, fp, fn, tp = cm.ravel()
print("TN", tn, " FP", fp, " FN", fn, " TP", tp)

**1a.** The blanks, in order: `0`, `0.25`, `y`, `X_train_full, y_train_full`, `fit_transform`, `transform`, `X_train_scaled, y_train`, `X_valid_scaled`. Sizes: 319 training, 107 validation, 143 test (put away).

**1b.** Validation set is 107 tumors, 40 malignant. Confusion matrix `[[65, 2], [2, 38]]`.

- TN 65: benign, and the model said benign. Patient goes home, correctly.
- FP 2: benign, but the model flagged it. Two patients get a follow up scan they did not need, and a scary week.
- FN 2: **malignant, and the model said benign.** Two patients with cancer were sent home. This is the cell that matters.
- TP 38: malignant, and the model caught it.

**1c.** Accuracy 0.963 against a baseline of 0.626. Unlike the wine, this model is doing a great deal: 34 points over doing nothing. The classes are much less imbalanced here (37 percent malignant), so accuracy is less misleading, but it still hides the 2 missed cancers.

---

## Problem 2: Precision, recall, and moving the threshold (steps outlined)

**What to copy, what to write:** Nothing to copy here. The steps are listed as comments in the cell. Write the line or two that does each step. Every one of them is a `LEARN TO WRITE THIS` cell in the notes.

**2a.** Precision, recall, and F1 at the default threshold. Which one do you care about most for a screening test, and what is its value?

**2b.** Get the probability of malignant for every validation tumor into a variable called `y_prob`. Print the first five, alongside `y_pred[:5]` and the truth, and check they agree with each other.

**2c.** Build your own predictions at a threshold of **0.1**. Print the confusion matrix, precision, and recall. Compare to 2a: how many malignant tumors does the model now miss, and what did it cost in false alarms?

**2d.** You are the doctor who has to choose the threshold. Write three or four sentences: which of 0.5 and 0.1 do you pick, what does each mistake cost the patient, and is there anything you would want to know before deciding.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

# Step 1 (2a)
print("precision:", round(precision_score(y_valid, y_pred), 3))
print("recall:   ", round(recall_score(y_valid, y_pred), 3))
print("f1:       ", round(f1_score(y_valid, y_pred), 3))

# Step 2 (2b)
y_prob = model.predict_proba(X_valid_scaled)[:, 1]
print()
print("P(malignant):", np.round(y_prob[:5], 3))
print("predict():   ", y_pred[:5])
print("truth:       ", y_valid.values[:5])

# Step 3 (2c)
threshold = 0.1
y_pred_10 = (y_prob >= threshold).astype(int)
print()
print(confusion_matrix(y_valid, y_pred_10))
print("precision at 0.1:", round(precision_score(y_valid, y_pred_10), 3))
print("recall at 0.1:   ", round(recall_score(y_valid, y_pred_10), 3))

**2a.** Precision 0.95, recall 0.95, F1 0.95. For screening, recall: it is the share of cancers we caught, and 0.95 means we missed 5 percent of them (2 of 40).

**2b.** `y_prob = model.predict_proba(X_valid_scaled)[:, 1]`. The first five probabilities are 0.000, 0.000, 0.988, 0.000, 0.163; predict gave 0, 0, 1, 0, 0 and the truth is 0, 0, 1, 0, **1**. The fifth tumor is a malignant one the model gave 0.163 and sent home. If you noticed that, you already understand 2c!

**2c.** At 0.1 the matrix is `[[56, 11], [1, 39]]`. Precision 0.78, recall 0.975. The model now misses **1** malignant tumor instead of 2, and the cost was 11 false alarms instead of 2.

**2d.** Your answer will be different, here is how I think about it. I would pick 0.1, or even lower. At 0.1 we catch one more cancer and pay for it with nine more scans that were not needed, and a scan is cheap next to a missed cancer. Things I would want to know before deciding: what happens to a flagged patient (a biopsy costs a lot more than a scan), how much worry a false alarm causes, and whether there is a threshold that catches all 40 malignant tumors. Picking 0.5 just because it is the default misses the point of today.

---

## Problem 3: ROC AUC and average precision (steps outlined)

**What to copy, what to write:** The plotting code is fine to copy from the notes and change the variable names. Write the two score lines yourself.

**3a.** Compute the ROC AUC and the average precision from `y_prob`. (Not from `y_pred`. If you pass `y_pred` it will run without complaint and give you a wrong number. Try it once so you have seen it.)

**3b.** Plot either the ROC curve or the precision/recall curve, your choice, with the random guessing line on it.

**3c.** On the wine data, AUC was 0.870 and AP was 0.532, a big gap. Here the two numbers are much closer together. Explain why, using the share of positives in each dataset.

**3d.** Suppose a colleague built a second model on this data with ROC AUC 0.995 (yours is a little lower). Does that mean their model will miss fewer cancers at the threshold you chose in 2d? Explain what AUC does and does not tell you.

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve

# Step 1 (3a)
auc = roc_auc_score(y_valid, y_prob)
ap = average_precision_score(y_valid, y_prob)
print("ROC AUC from y_prob:          ", round(auc, 3))
print("average precision from y_prob:", round(ap, 3))

# Step 2 (the wrong way)
print("ROC AUC from y_pred (wrong):          ", round(roc_auc_score(y_valid, y_pred), 3))
print("average precision from y_pred (wrong):", round(average_precision_score(y_valid, y_pred), 3))

# Step 3 (3b)
fpr, tpr, cutoffs = roc_curve(y_valid, y_prob)
plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, "b-", linewidth=2, label="logistic regression")
plt.plot([0, 1], [0, 1], "k--", linewidth=1, label="random guessing")
plt.xlabel("false positive rate")
plt.ylabel("true positive rate (recall)")
plt.title("ROC curve, AUC = " + str(round(auc, 3)))
plt.grid()
plt.legend()
plt.show()

**3a.** ROC AUC 0.990, average precision 0.987. From `y_pred` instead: 0.960 and 0.921. Both run, both are wrong, because a 0/1 vector has only one threshold in it and the curves collapse to a single point.

**3b.** Either curve is fine, as long as the random guessing line is on it. The ROC curve is very close to the top left corner.

**3c.** Here 37 percent of tumors are malignant, so positives and negatives are of similar size and the two denominators (negatives for the false positive rate, positive predictions for precision) are of similar size too. On the wine data only 13.6 percent were good, so the false positive rate was divided by 259 negatives and stayed tiny while precision, divided by a couple dozen yeses, did not. A big gap between AUC and AP is a sign of imbalanced classes.

**3d.** No. AUC scores the ranking across **all** thresholds and never looks at any one of them. Two models with AUC 0.990 and 0.995 could easily swap places at threshold 0.1. To know which misses fewer cancers at your threshold you have to look at the confusion matrix at that threshold. Use AUC to compare how well models rank the tumors, and use the confusion matrix at your threshold to make the actual decision.

---

## Problem 4: By hand, no code

A credit card company runs a fraud model on **10,000** transactions. **20** of them are actually fraud. The model flags **60** transactions, and **15** of those really are fraud.

**4a.** Fill in the full confusion matrix (all four cells) in sklearn's layout.

**4b.** Compute accuracy, precision, and recall. Show your arithmetic.

**4c.** What accuracy does a model that flags nothing get? Compare it to 4b and say in one sentence why accuracy is the wrong number here.

**4d.** The company is considering a second model that flags 200 transactions and catches all 20 frauds. Compute its precision and recall. Then say which model you would deploy, knowing that every flag freezes a customer's card until they call in.

**4a.** TP 15, FP 45, FN 5, TN 9935. `[[9935, 45], [5, 15]]`.

**4b.** Accuracy (15 + 9935) / 10000 = 0.995. Precision 15 / 60 = 0.25. Recall 15 / 20 = 0.75.

**4c.** Flag nothing: 9980 / 10000 = 0.998, which is **higher** than the model. Accuracy is the wrong number because the base rate is 99.8 percent, so a model that finds nothing beats a model that finds most of the fraud.

**4d.** Second model: precision 20 / 200 = 0.10, recall 20 / 20 = 1.0. It catches everything at the cost of 180 frozen cards per 10,000 transactions instead of 45. Either choice is fine if you explain it: the first model misses 5 frauds and annoys 45 customers, the second misses none and annoys 180. The important part is naming both costs. Saying "the second one because recall is 1.0" without thinking about the 180 customers is not enough.

---

## Problem 5: Catch every cancer, then open the test set once (just the task)

**What to copy, what to write:** All yours. Everything you need is in the notes and in Problems 1 to 3.

**5a.** On the **validation set**, find the **highest** threshold at which the model catches **every** malignant tumor (recall exactly 1.0). Report that threshold, and the precision and the number of false alarms that come with it. A loop over a list of thresholds is fine, and so is looking inside the arrays that `precision_recall_curve` returns.

**5b.** Out of every 100 benign patients, how many would get a false alarm at that threshold?

**5c.** Now, once, the **test set**. Scale `X_test` with the scaler you already have, get its probabilities, apply the threshold from 5a, and print the confusion matrix, precision, and recall. Did it still catch every malignant tumor? Then two or three sentences: why the validation set's promise of recall 1.0 did not *have* to hold on the test set, and why it was fine to look at the test set here but would not have been fine in 5a. (Hint: "The Test Set, Once" in the notes.)

In [ ]:
# 5a, on the validation set: walk down from 0.5 and stop at the first threshold with recall 1.0
thresholds = [0.5, 0.4, 0.3, 0.2, 0.1, 0.09, 0.08, 0.07, 0.06, 0.05, 0.04, 0.03, 0.02, 0.01]

for t in thresholds:
    y_pred_t = (y_prob >= t).astype(int)
    r = recall_score(y_valid, y_pred_t)
    p = precision_score(y_valid, y_pred_t)
    fp_count = ((y_pred_t == 1) & (y_valid.values == 0)).sum()
    print("threshold", t, "  recall", round(r, 3), "  precision", round(p, 3), "  false alarms", fp_count)

In [ ]:
# the exact answer: the smallest P(malignant) among the validation malignant tumors is the
# highest threshold that still catches all of them
chosen = y_prob[y_valid.values == 1].min()
print("highest threshold with recall 1.0 on validation:", round(chosen, 4))

y_pred_all = (y_prob >= chosen).astype(int)
print(confusion_matrix(y_valid, y_pred_all))
print("precision:", round(precision_score(y_valid, y_pred_all), 3))
false_alarms = ((y_pred_all == 1) & (y_valid.values == 0)).sum()   # flagged, but benign
n_benign = (y_valid.values == 0).sum()
print("false alarms:", false_alarms, "out of", n_benign, "benign patients")
print("false alarms per 100 benign patients:", round(100 * false_alarms / n_benign, 1))

In [ ]:
# 5c: the test set, once. Same scaler, same model, the threshold chosen on validation.
X_test_scaled = scaler.transform(X_test)
y_prob_test = model.predict_proba(X_test_scaled)[:, 1]
y_pred_test = (y_prob_test >= chosen).astype(int)

print("applied to the test set, once:")
print(confusion_matrix(y_test, y_pred_test))
print("recall:   ", round(recall_score(y_test, y_pred_test), 3))
print("precision:", round(precision_score(y_test, y_pred_test), 3))

**5a.** With the list of thresholds, 0.04 is the first one that gets recall 1.0 on validation: precision 0.667 and 20 false alarms. The exact answer is the smallest probability the model gave any validation malignant tumor, 0.0498, which gives precision 0.678 and 19 false alarms. Either answer is fine.

**5b.** 19 of 67 benign patients, about 28 per 100.

**5c.** At 0.0498 on the test set: `[[76, 14], [0, 53]]`, recall 1.0, precision 0.791. It did catch all 53 this time, with 14 false alarms out of 90 benign patients. It did not have to: the threshold was set by the single lowest-scoring malignant tumor in the validation set, and nothing guarantees the test set has no malignant tumor scoring lower than that. The promise of catching every single one came from one data point. Looking at the test set was fine here because every choice (the model, the threshold) was already made; it would not have been fine in 5a because there the number was being used to *make* the choice, and a test set that helps choose is just a second validation set.

This question ties Day 4 and Day 5 together, so it is worth taking your time on it!